# SQL — Syntax Reference

**Companion to:** `reference_sql_patterns.ipynb` — question patterns, templates, and decision guides

**Purpose:** quick lookup for function names, argument order, dialect differences, and syntax variants. Open this when you need to remember how something is written.

## Quick Index

| Section | Content |
| :--- | :--- |
| §1 | Core query structure & execution order |
| §2 | Aggregations |
| §3 | Conditional logic |
| §4 | Joins |
| §5 | Subqueries & CTEs |
| §6 | Window functions |
| §7 | Date & time |
| §8 | NULL handling |
| §9 | String functions |
| §10 | Regex & pattern matching |
| §11 | DELETE & UPDATE |
| §12 | Performance tips |

---
## §1 — Core Query Structure

```sql
SELECT   column1, AGG(column2)         -- 6: what to return
FROM     table_name                    -- 1: source
JOIN     other ON table.id = other.id  -- 2: combine
WHERE    condition                     -- 3: filter rows BEFORE grouping
GROUP BY column1                       -- 4: group
HAVING   AGG(column2) > 100            -- 5: filter AFTER grouping
ORDER BY column1 DESC                  -- 7: sort
LIMIT    10;                           -- 8: cap rows
```

**Execution order (not writing order):**
`FROM` → `JOIN` → `WHERE` → `GROUP BY` → `HAVING` → `SELECT` → `ORDER BY` → `LIMIT`

**Common mistakes:**
- Using a `SELECT` alias in `WHERE` — alias doesn't exist yet; use the expression directly or wrap in a subquery
- Filtering aggregated results with `WHERE` instead of `HAVING`
- Selecting a non-aggregated column not in `GROUP BY` — causes error in strict SQL modes

---
## §2 — Aggregations

### §2a — Basic aggregations

```sql
SELECT
  COUNT(*)                 AS total_rows,      -- includes NULLs
  COUNT(col)               AS non_null_rows,   -- excludes NULLs
  COUNT(DISTINCT user_id)  AS unique_users,
  SUM(sales)               AS total_sales,
  AVG(price)               AS avg_price,       -- NULLs excluded automatically
  MIN(score)               AS min_score,
  MAX(score)               AS max_score
FROM t;
```

**COUNT(\*) vs COUNT(col):** `COUNT(*)` counts all rows including NULLs; `COUNT(col)` counts only non-NULL values in that column.

### §2b — Conditional aggregation

```sql
-- Pivot rows into columns using CASE WHEN inside aggregate
SELECT
  region,
  SUM(CASE WHEN platform = 'iOS'     THEN revenue ELSE 0 END) AS ios_rev,
  SUM(CASE WHEN platform = 'Android' THEN revenue ELSE 0 END) AS android_rev,
  AVG(CASE WHEN platform = 'iOS'     THEN revenue END)        AS ios_avg  -- NULLs auto-excluded
FROM sales
GROUP BY region;
```

**ELSE 0 vs omitting ELSE:**
- `SUM(CASE WHEN ... THEN val ELSE 0 END)` — non-matching rows contribute 0 to the sum
- `SUM(CASE WHEN ... THEN val END)` — non-matching rows contribute NULL, which SUM ignores
- `AVG(CASE WHEN ... THEN val END)` — NULLs excluded from both numerator and denominator
- Use `SUM / COUNT(*)` when NULLs should be treated as 0 in AVG

### §2c — ROLLUP & GROUPING()

```sql
-- ROLLUP: adds subtotal and grand total rows automatically
SELECT region, platform, SUM(revenue) AS total
FROM sales
GROUP BY region, platform WITH ROLLUP;
-- Produces: per (region, platform), per region subtotal, grand total

-- GROUPING(): returns 1 for rollup rows, 0 for real data rows
SELECT
  CASE WHEN GROUPING(region) = 1 THEN 'ALL' ELSE region END AS region,
  SUM(revenue) AS total
FROM sales
GROUP BY region WITH ROLLUP;
```

**Common mistakes:**
- Using `AVG(col)` when NULLs should count as 0 — use `SUM(col) / COUNT(*)` instead
- Forgetting `ELSE 0` in `SUM(CASE WHEN ...)` — NULL propagates and sum becomes NULL
- Mistaking rollup NULL for missing data — use `GROUPING()` to distinguish

---
## §3 — Conditional Logic

### §3a — CASE WHEN (multi-condition, portable)

```sql
SELECT user_id,
  CASE
    WHEN spend > 100              THEN 'High'
    WHEN spend BETWEEN 50 AND 100 THEN 'Medium'
    ELSE 'Low'                    -- omitting ELSE returns NULL
  END AS spender_type
FROM transactions;
```

### §3b — IF() (binary condition, MySQL only)

```sql
SELECT user_id,
  IF(spend > 100, 'High', 'Low') AS spender_type
FROM transactions;
```

### §3c — NULLIF & IFNULL

```sql
NULLIF(a, b)       -- returns NULL if a = b, else returns a
                   -- useful to avoid division by zero: val / NULLIF(denom, 0)
IFNULL(col, 0)     -- returns 0 if col is NULL (MySQL shorthand)
```

**Decision — which to use:**
| Tool | Use when |
| :--- | :--- |
| `CASE WHEN` | Multiple conditions, any database |
| `IF()` | Binary yes/no, MySQL only |
| `COALESCE` | NULL fallback only — not a conditional |
| `NULLIF` | Convert a specific value to NULL |
| `IFNULL` | Single NULL replacement, MySQL shorthand |

**Common mistakes:**
- Conditions evaluated top to bottom — put most specific condition first
- Omitting `ELSE` — unmatched rows return NULL, not empty string
- Using `IF()` outside MySQL — use `CASE WHEN` for portability

---
## §4 — Joins

### §4a — Join types

```sql
-- INNER JOIN — only matching rows from both tables
SELECT * FROM A INNER JOIN B ON A.id = B.id;

-- LEFT JOIN — all rows from A, NULL for non-matching B
SELECT * FROM A LEFT JOIN B ON A.id = B.id;

-- FULL OUTER JOIN — all rows from both (MySQL: emulate with UNION)
SELECT * FROM A LEFT  JOIN B ON A.id = B.id
UNION
SELECT * FROM A RIGHT JOIN B ON A.id = B.id;

-- CROSS JOIN — every row of A × every row of B
SELECT * FROM A CROSS JOIN B;

-- Self join — join table to itself
SELECT a.employee_id, a.name, b.name AS manager
FROM employees a
LEFT JOIN employees b ON a.manager_id = b.employee_id;
```

### §4b — Anti-join: finding no-match rows

```sql
-- Method 1: LEFT JOIN + IS NULL (most common)
SELECT A.* FROM A
LEFT JOIN B ON A.id = B.id
WHERE B.id IS NULL;

-- Method 2: NOT EXISTS (clearest intent, handles NULLs correctly)
SELECT * FROM A
WHERE NOT EXISTS (SELECT 1 FROM B WHERE B.id = A.id);

-- Method 3: NOT IN (avoid if B.id can contain NULLs)
SELECT * FROM A
WHERE A.id NOT IN (SELECT id FROM B);
-- WARNING: if any value in B.id is NULL, NOT IN returns empty result
```

**Decision — anti-join method:**
| Method | Use when |
| :--- | :--- |
| `LEFT JOIN + IS NULL` | General purpose, widely supported |
| `NOT EXISTS` | Subquery references outer query; NULLs in B are safe |
| `NOT IN` | Only when B column is guaranteed NOT NULL |

### §4c — Filter placement: ON vs WHERE

```sql
-- ON + AND: filter BEFORE join — keeps all left rows
SELECT * FROM A
LEFT JOIN B ON A.id = B.id AND B.status = 'active';
-- Non-matching B rows appear as NULL — A rows are never lost

-- WHERE: filter AFTER join — silently converts LEFT JOIN to INNER JOIN
SELECT * FROM A
LEFT JOIN B ON A.id = B.id
WHERE B.status = 'active';  -- rows with no B match are now excluded
```

**Common mistakes:**
- `WHERE B.col = ...` on a LEFT JOIN silently converts it to INNER JOIN
- `NOT IN` with a subquery that can return NULLs — returns empty result
- Joining on columns with different data types — causes implicit cast, may skip index

---
## §5 — Subqueries & CTEs

### §5a — Inline subquery

```sql
-- Use for one-off filters or derived tables
SELECT user_id, score
FROM (
  SELECT user_id, score,
         DENSE_RANK() OVER (ORDER BY score DESC) AS rnk
  FROM Scores
) ranked
WHERE rnk = 2;
```

### §5b — Correlated subquery

```sql
-- References the outer query — runs once per outer row
-- Use for "latest record per group" or "max within group" patterns
SELECT e.*
FROM employees e
WHERE salary = (
  SELECT MAX(salary)
  FROM employees
  WHERE department_id = e.department_id  -- references outer row
);

-- Prefer JOIN or window function for performance on large tables
-- Correlated subquery runs once per row — O(n²) in worst case
```

### §5c — CTE (WITH clause)

```sql
-- Single CTE
WITH ranked AS (
  SELECT user_id, score,
         DENSE_RANK() OVER (ORDER BY score DESC) AS rnk
  FROM Scores
)
SELECT user_id, score FROM ranked WHERE rnk = 2;

-- Multiple chained CTEs
WITH
  cte1 AS (SELECT ...),
  cte2 AS (SELECT ... FROM cte1)   -- can reference earlier CTEs
SELECT * FROM cte2;
```

### §5d — Recursive CTE

```sql
-- Structure: anchor UNION ALL recursive step
WITH RECURSIVE cte AS (
  -- Anchor: base case, runs once
  SELECT id, name, manager_id, 0 AS depth
  FROM employees
  WHERE manager_id IS NULL

  UNION ALL  -- never UNION — deduplication breaks recursion

  -- Recursive step: runs until no new rows produced
  SELECT e.id, e.name, e.manager_id, c.depth + 1
  FROM employees e
  JOIN cte c ON e.manager_id = c.id
)
SELECT * FROM cte;

-- Date series generation
WITH RECURSIVE dates AS (
  SELECT '2024-01-01' AS dt
  UNION ALL
  SELECT DATE_ADD(dt, INTERVAL 1 DAY)
  FROM dates
  WHERE dt < '2024-01-31'          -- termination condition
)
SELECT * FROM dates;
```

**Decision — subquery vs CTE vs JOIN:**
| Tool | Use when |
| :--- | :--- |
| Inline subquery | One-off filter, used once, simple |
| Correlated subquery | Per-row lookup against outer query |
| CTE | Referenced more than once, or readability matters |
| Recursive CTE | Hierarchical data, date series generation |
| JOIN | Replace correlated subquery for better performance |

**Common mistakes:**
- Using `UNION` instead of `UNION ALL` in recursive CTE — deduplicates and breaks recursion
- Missing termination condition — causes infinite recursion
- Overusing CTEs for every step — unnecessary CTEs add overhead in some engines

---
## §6 — Window Functions

### §6a — Syntax skeleton

```sql
FUNCTION() OVER (
  PARTITION BY col   -- reset window per group (optional)
  ORDER BY col       -- ordering within window (required for ranking/lag/frames)
  ROWS BETWEEN ...   -- physical row frame (optional)
  -- or RANGE BETWEEN ... value-based frame
)
```

**PARTITION BY vs ORDER BY inside OVER():**
- `PARTITION BY` — resets the window for each group (like GROUP BY but without collapsing rows)
- `ORDER BY` — defines the order within each partition; required for LAG, LEAD, ranking, and frames
- Both optional — `OVER()` with no arguments applies the function across all rows

### §6b — Ranking functions

```sql
SELECT user_id, score,
  ROW_NUMBER()  OVER (ORDER BY score DESC) AS row_num,   -- unique, no ties
  RANK()        OVER (ORDER BY score DESC) AS rnk,       -- ties share rank, gaps after
  DENSE_RANK()  OVER (ORDER BY score DESC) AS dense_rnk, -- ties share rank, no gaps
  NTILE(4)      OVER (ORDER BY score DESC) AS quartile   -- divide into 4 equal buckets
FROM Scores;
```

| Function | Ties | Gaps | Use when |
| :--- | :--- | :--- | :--- |
| `ROW_NUMBER()` | No — always unique | n/a | Need unique identifier per row |
| `RANK()` | Yes — share rank | Yes | Standard competition ranking |
| `DENSE_RANK()` | Yes — share rank | No | Rank without gaps (most common in LeetCode) |
| `NTILE(n)` | Splits evenly | n/a | Percentile / quartile bucketing |

### §6c — LAG / LEAD / FIRST_VALUE / LAST_VALUE

```sql
-- LAG / LEAD: access previous / next row value
SELECT user_id, dt, amount,
  LAG(amount, 1)        OVER (PARTITION BY user_id ORDER BY dt) AS prev_amt,
  LAG(amount, 1, 0)     OVER (PARTITION BY user_id ORDER BY dt) AS prev_amt_default0,
  -- third argument = default when no previous row exists (avoids NULL at boundary)
  LEAD(amount, 1)       OVER (PARTITION BY user_id ORDER BY dt) AS next_amt
FROM payments;

-- FIRST_VALUE / LAST_VALUE: first or last in window
SELECT user_id, dt, amount,
  FIRST_VALUE(amount) OVER (
    PARTITION BY user_id ORDER BY dt
    ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING
  ) AS first_amt,
  LAST_VALUE(amount) OVER (
    PARTITION BY user_id ORDER BY dt
    ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING
    -- frame required — default frame stops at CURRENT ROW
  ) AS last_amt
FROM payments;

-- NTH_VALUE: nth row in the window
NTH_VALUE(amount, 2) OVER (PARTITION BY user_id ORDER BY dt
  ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING)
```

### §6d — Frames: ROWS BETWEEN vs RANGE BETWEEN

```sql
-- ROWS BETWEEN: physical row positions
-- Use for running totals, moving averages over last k records
SUM(amount) OVER (
  PARTITION BY user_id ORDER BY dt
  ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW  -- running total
)

AVG(amount) OVER (
  PARTITION BY user_id ORDER BY dt
  ROWS BETWEEN 6 PRECEDING AND CURRENT ROW          -- 7-row moving average
)

-- RANGE BETWEEN: value-based — rows with same ORDER BY value move together
-- Use for value-based windows or cumulative logic that respects ties
SUM(amount) OVER (
  ORDER BY amount
  RANGE BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW -- includes all rows with same amount
)

COUNT(*) OVER (
  ORDER BY amount
  RANGE BETWEEN 10 PRECEDING AND CURRENT ROW        -- rows within 10 units of current value
)
```

| Clause | Frame based on | Use for |
| :--- | :--- | :--- |
| `ROWS BETWEEN` | Physical row count | Running totals, last-k-row averages |
| `RANGE BETWEEN` | ORDER BY value | Value-based windows, tie-aware cumulative logic |

**Frame boundary keywords:**
- `UNBOUNDED PRECEDING` — from the first row of the partition
- `N PRECEDING` — N rows/values before current
- `CURRENT ROW` — the current row
- `N FOLLOWING` — N rows/values after current
- `UNBOUNDED FOLLOWING` — to the last row of the partition

### §6e — Other window functions

```sql
-- PERCENT_RANK: relative rank as fraction (0 to 1)
-- (rank - 1) / (total rows - 1)
PERCENT_RANK() OVER (ORDER BY score DESC)

-- CUME_DIST: cumulative distribution (0 to 1)
-- fraction of rows with value <= current
CUME_DIST() OVER (ORDER BY score DESC)

-- Share of group total
SELECT dept, emp_id, salary,
  salary * 1.0 / SUM(salary) OVER (PARTITION BY dept) AS pct_of_dept
FROM Employee;

-- Top N per group
WITH ranked AS (
  SELECT dept, emp_id, salary,
    ROW_NUMBER() OVER (PARTITION BY dept ORDER BY salary DESC) AS rn
  FROM Employee
)
SELECT * FROM ranked WHERE rn <= 3;

-- Kth highest without window functions
SELECT salary FROM Employee ORDER BY salary DESC LIMIT 1 OFFSET 1; -- 2nd highest
```

**Common mistakes:**
- Using `GROUP BY` with a window function on the same column — GROUP BY collapses rows; use CTE instead
- Omitting `ORDER BY` inside `OVER()` for LAG/LEAD — result is non-deterministic
- Forgetting `ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING` for `LAST_VALUE` — default frame stops at CURRENT ROW
- Using `RANGE` when you want exactly the last k records — use `ROWS` instead

---
## §7 — Date & Time

### §7a — Extraction

```sql
-- Date components
DATE(event_ts)                   -- date portion only: '2024-04-01'
TIME(event_ts)                   -- time portion only: '14:30:00'
EXTRACT(YEAR  FROM event_ts)     -- year (portable)
EXTRACT(MONTH FROM event_ts)     -- month 1–12
EXTRACT(DAY   FROM event_ts)     -- day of month
YEAR(event_ts)                   -- MySQL shorthand
MONTH(event_ts)
DAY(event_ts)

-- Time components
HOUR(event_ts)                   -- 0–23
MINUTE(event_ts)                 -- 0–59
SECOND(event_ts)                 -- 0–59

-- Week / day of week
WEEK(date)                       -- week number 0–53
YEARWEEK(date)                   -- YYYYWW format — avoids cross-year ambiguity
DAYOFWEEK(date)                  -- 1=Sunday, 7=Saturday (MySQL)
WEEKDAY(date)                    -- 0=Monday, 6=Sunday (MySQL) — different numbering!
EXTRACT(DOW FROM date)           -- 0=Sunday, 6=Saturday (PostgreSQL)
```

### §7b — Formatting & Truncation

```sql
-- DATE_FORMAT (MySQL)
DATE_FORMAT(event_ts, '%Y-%m')       -- '2024-04'
DATE_FORMAT(event_ts, '%Y-%m-%d')    -- '2024-04-01'
DATE_FORMAT(event_ts, '%Y-%u')       -- '2024-14' (year-week)

-- Common format specifiers:
-- %Y = 4-digit year   %y = 2-digit year
-- %m = month 01-12    %M = month name
-- %d = day 01-31      %D = day with suffix (1st, 2nd)
-- %H = hour 00-23     %h = hour 01-12
-- %i = minutes        %s = seconds
-- %W = weekday name   %u = week number (Monday start)

-- Truncation to start of period (MySQL)
DATE_FORMAT(date, '%Y-%m-01')        -- start of month
DATE_FORMAT(date, '%Y-01-01')        -- start of year

-- Truncation (PostgreSQL)
DATE_TRUNC('month', event_ts)        -- start of month
DATE_TRUNC('week',  event_ts)        -- start of week
DATE_TRUNC('year',  event_ts)        -- start of year

-- String → Date conversion
STR_TO_DATE('2024-04-01', '%Y-%m-%d')  -- MySQL: string to date
TO_DATE('2024-04-01', 'YYYY-MM-DD')    -- PostgreSQL equivalent
CAST('2024-04-01' AS DATE)             -- portable casting
CONVERT('2024-04-01', DATE)            -- MySQL alternative
```

### §7c — Arithmetic

```sql
-- Add / subtract intervals
DATE_ADD(date, INTERVAL 7 DAY)       -- add 7 days
DATE_ADD(date, INTERVAL 1 MONTH)     -- add 1 month
DATE_ADD(date, INTERVAL 1 YEAR)
DATE_SUB(date, INTERVAL 30 DAY)      -- subtract 30 days
date + INTERVAL '7 days'             -- PostgreSQL syntax

-- Difference between dates
DATEDIFF(end_date, start_date)       -- days only (MySQL: end − start)
end_date - start_date                -- PostgreSQL: returns interval

-- TIMESTAMPDIFF — difference in any unit (more flexible than DATEDIFF)
TIMESTAMPDIFF(SECOND, start_ts, end_ts)   -- seconds between
TIMESTAMPDIFF(MINUTE, start_ts, end_ts)   -- minutes between
TIMESTAMPDIFF(HOUR,   start_ts, end_ts)   -- hours between
TIMESTAMPDIFF(DAY,    start_ts, end_ts)   -- days between (same as DATEDIFF)
TIMESTAMPDIFF(MONTH,  start_ts, end_ts)   -- months between
TIMESTAMPDIFF(YEAR,   start_ts, end_ts)   -- years between

-- TIMEDIFF — difference between two time/datetime values → returns TIME
TIMEDIFF('14:30:00', '12:00:00')     -- returns '02:30:00'
```

**DATEDIFF vs TIMESTAMPDIFF:**
- `DATEDIFF` — days only, argument order is `(end, start)` in MySQL
- `TIMESTAMPDIFF` — any unit, argument order is `(unit, start, end)` — note reversed!

### §7d — Current date/time

```sql
CURRENT_DATE                         -- date only, no time
CURRENT_TIMESTAMP                    -- date + time, portable
NOW()                                -- MySQL alias for CURRENT_TIMESTAMP
SYSDATE()                            -- MySQL: time of execution (vs query start for NOW())
```

### §7e — Unix timestamp conversion

```sql
-- Identify: 10 digits = seconds, 13 digits = milliseconds
FROM_UNIXTIME(ts_col)                -- seconds → datetime (MySQL)
DATE(FROM_UNIXTIME(ts_col))          -- seconds → date only (MySQL)
FROM_UNIXTIME(ts_col / 1000)         -- milliseconds → datetime (MySQL)
TO_TIMESTAMP(ts_col)                 -- seconds → timestamptz (PostgreSQL)
TO_TIMESTAMP(ts_col)::DATE           -- seconds → date only (PostgreSQL)
```

**Common mistakes:**
- `DATEDIFF` vs `TIMESTAMPDIFF` argument order — MySQL `DATEDIFF(end, start)` vs `TIMESTAMPDIFF(unit, start, end)`
- `DAYOFWEEK` vs `WEEKDAY` numbering — `DAYOFWEEK` starts at 1=Sunday; `WEEKDAY` starts at 0=Monday
- Using `DATE_FORMAT` (MySQL-only) in PostgreSQL — use `TO_CHAR(date, 'YYYY-MM')` instead
- Forgetting to cast string columns before date arithmetic — implicit cast may cause wrong results

---
## §8 — NULL Handling

### §8a — Detection

```sql
SELECT * FROM t WHERE col IS NULL;
SELECT * FROM t WHERE col IS NOT NULL;
-- Never use = NULL or != NULL — always evaluates to unknown
```

### §8b — Replacement functions

```sql
COALESCE(a, b, c)      -- returns first non-NULL — portable, any number of args
IFNULL(col, 0)         -- returns 0 if NULL — MySQL shorthand for 2-arg COALESCE
NULLIF(a, b)           -- returns NULL if a = b, else a
                       -- classic use: NULLIF(denominator, 0) to avoid division by zero

-- Chained COALESCE (equivalent to SQL COALESCE)
SELECT COALESCE(phone, email, 'Unknown') AS contact FROM people;
```

**Decision — which to use:**
| Tool | Use when |
| :--- | :--- |
| `COALESCE` | Multiple fallbacks, any database |
| `IFNULL` | Single fallback, MySQL only |
| `NULLIF` | Convert a specific value to NULL |

### §8c — NULL behavior in aggregations & joins

```sql
COUNT(*)        -- includes NULLs
COUNT(col)      -- excludes NULLs
SUM(col)        -- ignores NULLs; returns NULL if ALL values are NULL
AVG(col)        -- ignores NULLs in numerator AND denominator
                -- use SUM(col) / COUNT(*) to treat NULLs as 0
MIN / MAX       -- ignores NULLs
```

**NULL in JOINs:** `NULL = NULL` is never true in SQL — two NULL join keys will never match.

**Common mistakes:**
- `WHERE col = NULL` — always false; use `IS NULL`
- `AVG` treating NULLs as 0 — it ignores them; use `SUM / COUNT(*)` to include NULLs as 0
- `NOT IN` with a subquery that returns NULLs — entire result becomes empty

---
## §9 — String Functions

### §9a — Extraction

```sql
SUBSTRING(str, start, length)    -- extract substring (1-indexed)
SUBSTR(str, start, length)       -- alias for SUBSTRING
LEFT(str, n)                     -- first n characters
RIGHT(str, n)                    -- last n characters
MID(str, start, length)          -- MySQL alias for SUBSTRING

-- Examples
SUBSTRING('Hello World', 1, 5)   -- 'Hello'
LEFT('Hello World', 5)           -- 'Hello'
RIGHT('Hello World', 5)          -- 'World'
```

### §9b — Search & position

```sql
INSTR(str, substr)               -- position of first occurrence (1-indexed, 0 if not found)
LOCATE(substr, str)              -- same as INSTR but argument order reversed
POSITION(substr IN str)          -- SQL standard syntax
CHARINDEX(substr, str)           -- SQL Server syntax

-- LIKE patterns (simpler than REGEXP)
col LIKE '%pattern%'             -- contains
col LIKE 'pattern%'              -- starts with
col LIKE '%pattern'              -- ends with
col LIKE 'p_ttern'               -- _ matches exactly one character
col LIKE 'p\_ttern' ESCAPE '\'  -- escape literal underscore
```

### §9c — Manipulation

```sql
CONCAT(a, b, c)                  -- join strings; returns NULL if any arg is NULL
CONCAT_WS('-', a, b, c)          -- join with separator; skips NULLs automatically
REPLACE(str, old, new)           -- replace all occurrences of old with new
TRIM(str)                        -- remove leading and trailing spaces
LTRIM(str)                       -- remove leading spaces only
RTRIM(str)                       -- remove trailing spaces only
TRIM(BOTH 'x' FROM str)          -- remove specific character from both ends
REVERSE(str)                     -- reverse the string
REPEAT(str, n)                   -- repeat string n times
LPAD(str, n, pad)                -- left-pad to length n with pad character
RPAD(str, n, pad)                -- right-pad to length n
```

### §9d — Inspection

```sql
LENGTH(str)                      -- byte length (differs for multibyte chars)
CHAR_LENGTH(str)                 -- character length — use this for Unicode
UPPER(str)                       -- convert to uppercase
LOWER(str)                       -- convert to lowercase
```

**Decision — LIKE vs REPLACE vs REGEXP:**
| Tool | Use when |
| :--- | :--- |
| `LIKE` | Simple prefix/suffix/contains, no special patterns |
| `REPLACE` | Exact substring substitution, no pattern needed |
| `REGEXP` | Complex format validation, character class matching |

**Common mistakes:**
- `CONCAT` returning NULL when any argument is NULL — use `CONCAT_WS` to skip NULLs
- `LENGTH` vs `CHAR_LENGTH` — for multibyte characters (e.g. UTF-8 Chinese), `LENGTH` returns bytes not characters
- `LIKE '%val%'` preventing index use — leading wildcard causes full table scan

---
## §10 — Regex & Pattern Matching

```sql
-- REGEXP / REGEXP_LIKE — returns 1 if match, 0 if not
SELECT * FROM users WHERE email REGEXP '^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$';
SELECT * FROM users WHERE REGEXP_LIKE(email, '^[a-zA-Z0-9][^@]+@leetcode\.com$');

-- REGEXP_REPLACE — replace matched portion
SELECT REGEXP_REPLACE(phone, '[^0-9]', '') AS digits_only FROM users;

-- REGEXP_SUBSTR — extract first matching substring
SELECT REGEXP_SUBSTR(description, '[0-9]+') AS first_number FROM t;

-- REGEXP_INSTR — position of first match (1-indexed, 0 if no match)
SELECT REGEXP_INSTR(col, '[A-Z]') AS first_upper_pos FROM t;
```

**Common LeetCode regex patterns:**

| Pattern | Regex |
| :--- | :--- |
| Valid email | `^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$` |
| Starts with letter | `^[A-Za-z]` |
| Digits only | `^[0-9]+$` |
| Alphanumeric only | `^[A-Za-z0-9]+$` |
| Exactly N characters | `^.{n}$` |

**Common anchors and quantifiers:**

| Symbol | Meaning |
| :--- | :--- |
| `^` | Start of string |
| `$` | End of string |
| `.` | Any single character |
| `*` | 0 or more |
| `+` | 1 or more |
| `?` | 0 or 1 |
| `{n}` | Exactly n |
| `[abc]` | Any one of a, b, c |
| `[^abc]` | Any except a, b, c |

**Common mistakes:**
- Forgetting `^` and `$` — without anchors, pattern matches anywhere in the string
- Using `\d` instead of `[0-9]` — `\d` not supported in all MySQL versions
- Escaping dots: `\.` must be `\\.` in MySQL string literals to match a literal dot

---
## §11 — DELETE & UPDATE Patterns

### §11a — DELETE

```sql
-- Basic DELETE with WHERE
DELETE FROM t WHERE condition;

-- DELETE with subquery
DELETE FROM users
WHERE id NOT IN (
  SELECT MIN(id) FROM users GROUP BY email  -- keep one row per email
);

-- DELETE with self-join — remove duplicates, keep row with smallest id
DELETE p1 FROM Person p1
JOIN Person p2
  ON p1.email = p2.email
 AND p1.id > p2.id;   -- delete the one with larger id
```

### §11b — UPDATE

```sql
-- UPDATE with CASE WHEN
UPDATE salary
SET sex = CASE
  WHEN sex = 'm' THEN 'f'
  WHEN sex = 'f' THEN 'm'
END;

-- UPDATE with JOIN (MySQL)
UPDATE orders o
JOIN customers c ON o.customer_id = c.id
SET o.discount = 0.1
WHERE c.tier = 'Gold';
```

**Decision — DELETE self-join vs subquery:**
| Method | Use when |
| :--- | :--- |
| Self-join DELETE | Comparing rows within the same table (duplicate removal) |
| Subquery DELETE | Filtering based on aggregate or derived condition |

**Common mistakes:**
- `DELETE` referencing the same table in a subquery — MySQL doesn't allow this directly; wrap in a derived table
- Omitting `WHERE` in DELETE/UPDATE — deletes/updates all rows
- `UPDATE` with `JOIN` syntax differs by database — PostgreSQL uses `FROM` clause instead

---
## §12 — Performance Tips

```sql
-- Filter early to reduce rows before joins and aggregations
SELECT * FROM orders
WHERE order_date >= '2024-01-01'     -- filter before joining
JOIN customers ON orders.customer_id = customers.id;

-- Inspect query plan
EXPLAIN SELECT ...;
EXPLAIN ANALYZE SELECT ...;          -- PostgreSQL: runs the query too

-- Avoid functions on indexed columns in WHERE
WHERE YEAR(created_at) = 2024        -- bad: full table scan
WHERE created_at >= '2024-01-01'
  AND created_at <  '2025-01-01'     -- good: index usable
```

- Avoid `SELECT *` — select only needed columns
- Index columns used in `JOIN ON`, `WHERE`, and `ORDER BY`
- Avoid correlated subqueries in `SELECT` — runs once per row; replace with JOIN or CTE
- `COUNT(DISTINCT ...)` is expensive on large tables — consider HLL approximation in analytical engines
- `UNION ALL` over `UNION` unless deduplication is explicitly needed
- Avoid leading wildcard `LIKE '%val'` — prevents index use; use full-text search for contains queries